In [20]:
import time
from pathlib import Path

import requests
import pandas as pd

In [21]:
BASE = "https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities"
START_DATE = "2014-01-01"

TICKERS = [
    "SBER", "GAZP", "LKOH", "GMKN", "NVTK", "ROSN", "TATN", "PLZL", "SNGSP",
    "X5", "MGNT", "CHMF", "NLMK", "ALRS", "AFLT", "IRAO", "RTKM", "MOEX",
    "PHOR", "VTBR", "SIBN", "SMLT", "POSI", "MAGN", "T",
]

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

In [22]:
def load_history(ticker, start_date=START_DATE):
    """Качает дневную историю по бумаге с MOEX ISS, обходя пагинацию."""
    rows, columns, cursor = [], None, 0
    while True:
        response = requests.get(
            f"{BASE}/{ticker}.json",
            params={"from": start_date, "start": cursor, "iss.meta": "off"},
            timeout=30,
        )
        response.raise_for_status()
        block = response.json()["history"]
        if not block["data"]:
            break
        columns = block["columns"]
        rows.extend(block["data"])
        cursor += len(block["data"])
    df = pd.DataFrame(rows, columns=columns)
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
    return df

In [23]:
sber = load_history("SBER")

print(sber.shape)
print(sber["TRADEDATE"].min(), sber["TRADEDATE"].max())
print("пропусков в CLOSE:", sber["CLOSE"].isna().sum())

sber[["TRADEDATE", "SECID", "CLOSE", "VOLUME"]].head()

(3190, 24)
2014-01-06 00:00:00 2026-08-19 00:00:00
пропусков в CLOSE: 18


,TRADEDATE,SECID,CLOSE,VOLUME
0,2014-01-06,SBER,98.91,31691800
1,2014-01-08,SBER,98.19,42372290
2,2014-01-09,SBER,98.00,45986900
3,2014-01-10,SBER,99.20,51902400
4,2014-01-13,SBER,100.25,62051250


### Пропуски в данных

18 дней без цены закрытия у SBER: остановка торгов на МосБирже
(конец февраля — март 2022) и отдельные праздничные дни.
Во всех случаях VOLUME = 0 и NUMTRADES = 0 — торгов не было.

Решение: строки без CLOSE удаляем. Заполнять их последней
известной ценой нельзя — это создало бы несуществующие
наблюдения и исказило распределение доходностей по дням недели.

In [24]:
missing = sber[sber["CLOSE"].isna()]
missing[["TRADEDATE", "CLOSE", "VOLUME", "NUMTRADES"]]

,TRADEDATE,CLOSE,VOLUME,NUMTRADES
2019,2022-01-07,NaN,0,0
2052,2022-02-23,NaN,0,0
2055,2022-02-28,NaN,0,0
2056,2022-03-01,NaN,0,0
2057,2022-03-02,NaN,0,0
2058,2022-03-03,NaN,0,0
2059,2022-03-04,NaN,0,0
2060,2022-03-09,NaN,0,0
2061,2022-03-10,NaN,0,0
2062,2022-03-11,NaN,0,0


In [25]:
sber["CLOSE"].isna().head(10)

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: CLOSE, dtype: bool

In [26]:
print(len(sber["CLOSE"]), len(sber["CLOSE"].isna()))

3190 3190


In [27]:
mask = sber["CLOSE"].isna()
sber[mask]

,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,...,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
2019,TQBR,2022-01-07,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,293.86,...,291.69,293.86,1.676223e+10,1951406.1,1.676223e+10,0.0,3,SUR,NaN,NaN
2052,TQBR,2022-02-23,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,208.38,...,211.00,208.38,1.295735e+11,567590.0,1.295735e+11,0.0,3,SUR,NaN,NaN
2055,TQBR,2022-02-28,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.631605e+10,1663875.0,4.631605e+10,0.0,3,SUR,NaN,NaN
2056,TQBR,2022-03-01,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.631605e+10,1663875.0,4.631605e+10,0.0,3,SUR,NaN,NaN
2057,TQBR,2022-03-02,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,1.116498e+11,1663875.0,1.116498e+11,0.0,3,SUR,NaN,NaN
2058,TQBR,2022-03-03,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.631605e+10,1663875.0,4.631605e+10,0.0,3,SUR,NaN,NaN
2059,TQBR,2022-03-04,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.070000e+11,1663875.0,4.070000e+11,0.0,3,SUR,NaN,NaN
2060,TQBR,2022-03-09,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,2.412234e+11,1663875.0,2.412234e+11,0.0,3,SUR,NaN,NaN
2061,TQBR,2022-03-10,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,1.116498e+11,1663875.0,1.116498e+11,0.0,3,SUR,NaN,NaN
2062,TQBR,2022-03-11,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,1.116498e+11,1663875.0,1.116498e+11,0.0,3,SUR,NaN,NaN


In [28]:
r = requests.get(
    f"{BASE}/SBER.json",
    params={"from": "2014-01-01", "start": 0, "iss.meta": "off"},
    timeout=30,
)

print(type(r))
print(r.status_code)
print(r.url)
print(len(r.text), "символов в ответе")

<class 'requests.models.Response'>
200
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/SBER.json?from=2014-01-01&start=0&iss.meta=off
21700 символов в ответе


In [29]:
r.text[:2000]

'{\n"history": {\n\t"columns": ["BOARDID", "TRADEDATE", "SHORTNAME", "SECID", "NUMTRADES", "VALUE", "OPEN", "LOW", "HIGH", "LEGALCLOSEPRICE", "WAPRICE", "CLOSE", "VOLUME", "MARKETPRICE2", "MARKETPRICE3", "ADMITTEDQUOTE", "MP2VALTRD", "MARKETPRICE3TRADESVALUE", "ADMITTEDVALUE", "WAVAL", "TRADINGSESSION", "CURRENCYID", "TRENDCLSPR", "TRADE_SESSION_DATE"], \n\t"data": [\n\t\t["TQBR", "2014-01-06", "Сбербанк", "SBER", 22830, 3154470383.9, 100.2, 98.62, 100.31, 98.63, 99.54, 98.91, 31691800, 99.54, 99.54, 99.54, 3156738969.34, 3156738969.34, 3156738969.34, null, 3, "SUR", -2.23, null],\n\t\t["TQBR", "2014-01-08", "Сбербанк", "SBER", 35633, 4179938515.5, 99.1, 97.85, 99.41, 98.2, 98.65, 98.19, 42372290, 98.65, 98.65, 98.65, 4182984403.36, 4182984403.36, 4182984403.36, null, 3, "SUR", -0.73, null],\n\t\t["TQBR", "2014-01-09", "Сбербанк", "SBER", 41567, 4518388781.2, 98.44, 97.69, 98.77, 97.97, 98.25, 98, 45986900, 98.25, 98.25, 98.25, 4526413769.96, 4526413769.96, 4526413769.96, null, 3, "SUR

In [30]:
data = r.json()
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['history', 'history.cursor'])


In [31]:
block = data["history"]
print(block.keys())
print(len(block["columns"]))
print(len(block["data"]))

dict_keys(['columns', 'data'])
24
100


In [32]:
print(block["columns"])
print(block["data"][0])

['BOARDID', 'TRADEDATE', 'SHORTNAME', 'SECID', 'NUMTRADES', 'VALUE', 'OPEN', 'LOW', 'HIGH', 'LEGALCLOSEPRICE', 'WAPRICE', 'CLOSE', 'VOLUME', 'MARKETPRICE2', 'MARKETPRICE3', 'ADMITTEDQUOTE', 'MP2VALTRD', 'MARKETPRICE3TRADESVALUE', 'ADMITTEDVALUE', 'WAVAL', 'TRADINGSESSION', 'CURRENCYID', 'TRENDCLSPR', 'TRADE_SESSION_DATE']
['TQBR', '2014-01-06', 'Сбербанк', 'SBER', 22830, 3154470383.9, 100.2, 98.62, 100.31, 98.63, 99.54, 98.91, 31691800, 99.54, 99.54, 99.54, 3156738969.34, 3156738969.34, 3156738969.34, None, 3, 'SUR', -2.23, None]


In [33]:
print(data["history.cursor"])

{'columns': ['INDEX', 'TOTAL', 'PAGESIZE'], 'data': [[0, 3190, 100]]}


In [34]:
frames = []

for i, ticker in enumerate(TICKERS, 1):
    df = load_history(ticker)
    frames.append(df)
    print(f"{i}/{len(TICKERS)} {ticker}: {len(df)} строк")
    time.sleep(0.5)

prices = pd.concat(frames, ignore_index=True)
print(prices.shape)

1/25 SBER: 3190 строк
2/25 GAZP: 3084 строк
3/25 LKOH: 3190 строк
4/25 GMKN: 3084 строк
5/25 NVTK: 3190 строк
6/25 ROSN: 3084 строк
7/25 TATN: 3190 строк
8/25 PLZL: 3084 строк
9/25 SNGSP: 3084 строк
10/25 X5: 410 строк
11/25 MGNT: 3190 строк
12/25 CHMF: 3084 строк
13/25 NLMK: 3084 строк
14/25 ALRS: 3190 строк
15/25 AFLT: 3190 строк
16/25 IRAO: 3180 строк
17/25 RTKM: 3190 строк
18/25 MOEX: 3190 строк
19/25 PHOR: 3190 строк
20/25 VTBR: 3190 строк
21/25 SIBN: 3084 строк
22/25 SMLT: 1474 строк
23/25 POSI: 1185 строк
24/25 MAGN: 3084 строк
25/25 T: 437 строк
(69532, 24)


In [35]:
#frames

In [36]:
prices

,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,...,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
0,TQBR,2014-01-06,Сбербанк,SBER,22830,3.154470e+09,100.20,98.62,100.31,98.63,...,99.54,99.54,3.156739e+09,3.156739e+09,3156738969.34,NaN,3,SUR,-2.23,NaN
1,TQBR,2014-01-08,Сбербанк,SBER,35633,4.179939e+09,99.10,97.85,99.41,98.20,...,98.65,98.65,4.182984e+09,4.182984e+09,4182984403.36,NaN,3,SUR,-0.73,NaN
2,TQBR,2014-01-09,Сбербанк,SBER,41567,4.518389e+09,98.44,97.69,98.77,97.97,...,98.25,98.25,4.526414e+09,4.526414e+09,4526413769.96,NaN,3,SUR,-0.19,NaN
3,TQBR,2014-01-10,Сбербанк,SBER,38198,5.109679e+09,97.87,97.52,99.41,99.41,...,98.45,98.45,5.113831e+09,5.113831e+09,5113831362.53,NaN,3,SUR,1.22,NaN
4,TQBR,2014-01-13,Сбербанк,SBER,29942,6.191507e+09,99.30,99.04,100.35,100.24,...,99.78,99.78,6.191738e+09,6.191738e+09,6191738491.4,NaN,3,SUR,1.06,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69527,TQBR,2026-08-13,Т-Техно ао,T,130628,7.053150e+09,277.98,264.14,279.16,268.00,...,273.02,None,4.939137e+09,4.939137e+09,None,0.0,3,SUR,-4.69,2026-08-13
69528,TQBR,2026-08-14,Т-Техно ао,T,169980,1.214493e+10,264.68,253.42,269.92,255.30,...,258.86,None,1.017689e+10,1.017689e+10,None,0.0,3,SUR,-3.64,2026-08-14
69529,TQBR,2026-08-17,Т-Техно ао,T,138466,8.971053e+09,255.40,248.78,255.40,250.82,...,251.56,None,4.962482e+09,4.962482e+09,None,0.0,3,SUR,-0.94,2026-08-17
69530,TQBR,2026-08-18,Т-Техно ао,T,111772,7.585771e+09,252.68,251.70,261.88,259.50,...,257.96,None,5.983526e+09,5.983526e+09,None,0.0,3,SUR,3.30,2026-08-18


In [37]:
summary = prices.groupby("SECID")["TRADEDATE"].agg(["min", "max", "count"])
summary.sort_values("count")

,min,max,count
SECID,,,
X5,2025-01-09,2026-08-19,410
T,2024-11-28,2026-08-19,437
POSI,2021-12-17,2026-08-19,1185
SMLT,2020-10-29,2026-08-19,1474
CHMF,2014-06-09,2026-08-19,3084
GAZP,2014-06-09,2026-08-19,3084
GMKN,2014-06-09,2026-08-19,3084
MAGN,2014-06-09,2026-08-19,3084
SNGSP,2014-06-09,2026-08-19,3084


In [38]:
RAW.mkdir(parents=True, exist_ok=True)

path = RAW / "moex_prices.csv"
prices.to_csv(path, index=False)

#print(path)

print(f"{path.stat().st_size / 1024**2:.1f} МБ")

11.4 МБ


### Состав выборки

- Все 25 бумаг торгуются по последний день выборки, делистинга нет.
- Четыре бумаги с короткой историей: X5 (407 дней), T (434),
  POSI (1182), SMLT (1471) — недавние IPO и смены тикера.
- Девять бумаг начинаются с 2014-06-09: перевод на режим Т+2,
  до этой даты они торговались в другом режиме торгов,
  которого нет в нашей выгрузке.

Решение: оставляем все бумаги. Единица наблюдения —
«бумага в конкретный день», разная длина истории её не искажает.
Следствие: поздние годы представлены большим числом бумаг,
поэтому устойчивость результата проверяем отдельно по подпериодам.